# Non-Leak OOF Stacking + CatBoost OOF + Blend + Seed Ensemble

This notebook rebuilds the pipeline with:
- leak-free stacking (TargetEncoder inside each base model pipeline)
- unified OOF CV framework
- seed ensemble
- OOF blend + submission generation


In [ ]:
# If needed, install dependencies in Kaggle (uncomment):
# !pip install -q scikit-learn==1.3.2 xgboost lightgbm catboost

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.preprocessing import TargetEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, CatBoostError

from scipy.stats import rankdata

import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message="X does not have valid feature names, but LGBMClassifier was fitted with feature names")


In [ ]:
# === Config ===
SEEDS = [42, 2024, 777]
N_SPLITS = 5
ENCODER_MODE = "target"  # "target" or "onehot"
USE_FEATURE_ENGINEERING = False
BLEND_GRID_STEP = 0.1

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/kaggle/input/playground-series-s6e2")
if not (DATA_DIR / "train.csv").exists():
    DATA_DIR = Path("data/raw")

print("DATA_DIR:", DATA_DIR.resolve())


In [ ]:
# === Data Load ===
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sub_path = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

# set index if id exists
if "id" in train.columns:
    train = train.set_index("id")
if "id" in test.columns:
    test = test.set_index("id")
if "id" in sub.columns:
    sub = sub.set_index("id")

TARGET = "Heart Disease"

# label-encode target only if needed
if train[TARGET].dtype == object:
    le = LabelEncoder()
    train[TARGET] = le.fit_transform(train[TARGET])

print("train:", train.shape, "test:", test.shape, "sub:", sub.shape)


In [ ]:
# === Feature Engineering (optional) ===

def apply_feature_engineering(df):
    df = df.copy()
    df['bp_chol_index'] = df['BP'] * df['Cholesterol'] / 100.0
    df['hr_age_ratio'] = df['Max HR'] / (df['Age'] + 1)
    df['st_slope_interaction'] = df['ST depression'] * df['Slope of ST']
    df['vessel_thal_score'] = df['Number of vessels fluro'] * df['Thallium']
    df['heart_stress_index'] = (df['BP'] + df['ST depression'] * 10) / (df['Max HR'] + 1)
    df['is_hypertensive'] = (df['BP'] > 140).astype(int)
    df['high_cholesterol'] = (df['Cholesterol'] > 240).astype(int)
    df['is_elderly'] = (df['Age'] > 60).astype(int)
    df['double_product'] = df['BP'] * df['Max HR']
    df['hr_reserve_pct'] = df['Max HR'] / (220 - df['Age'])
    df['sugar_pressure_inter'] = df['FBS over 120'] * df['BP']
    df['effort_pain_impact'] = df['Chest pain type'] * df['Exercise angina']
    df['risk_factor_count'] = (
        (df['BP'] > 130).astype(int) +
        (df['Cholesterol'] > 200).astype(int) +
        (df['FBS over 120']).astype(int) +
        (df['is_elderly'])
    )
    df['exercise_st_warning'] = ((df['ST depression'] > 0) & (df['Exercise angina'] == 1)).astype(int)
    df['abnormal_ekg_flag'] = (df['EKG results'] > 0).astype(int)
    return df

if USE_FEATURE_ENGINEERING:
    train = apply_feature_engineering(train)
    test = apply_feature_engineering(test)


In [ ]:
# === Helpers ===

def build_preprocessor(encoder_mode, cat_cols, num_cols):
    if encoder_mode == "target":
        try:
            enc = TargetEncoder(handle_unknown="value", handle_missing="value")
        except TypeError:
            # older sklearn TargetEncoder signature
            enc = TargetEncoder()
        transformer = ColumnTransformer(
            [("target_enc", enc, cat_cols)],
            remainder="passthrough",
        )
        return Pipeline([
            ("encoder", transformer),
            ("scaler", StandardScaler())
        ])
    elif encoder_mode == "onehot":
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        transformer = ColumnTransformer(
            [("onehot", enc, cat_cols)],
            remainder="passthrough",
        )
        return Pipeline([
            ("encoder", transformer),
            ("scaler", StandardScaler())
        ])
    else:
        raise ValueError("encoder_mode must be 'target' or 'onehot'")


def build_stacking_model(seed, encoder_mode, cat_cols, num_cols):
    preprocessor = build_preprocessor(encoder_mode, cat_cols, num_cols)

    xgb_params = {
        "n_estimators": 1200,
        "learning_rate": 0.04,
        "max_depth": 5,
        "min_child_weight": 5,
        "subsample": 0.97,
        "colsample_bytree": 0.516,
        "gamma": 0.157,
        "random_state": seed,
        "tree_method": "hist",
        "objective": "binary:logistic",
        "eval_metric": "logloss",
    }

    lgbm_params = {
        "n_estimators": 1100,
        "learning_rate": 0.1,
        "num_leaves": 151,
        "max_depth": 3,
        "min_child_samples": 86,
        "subsample": 0.93,
        "colsample_bytree": 0.516,
        "random_state": seed,
        "objective": "binary",
        "verbosity": -1,
    }

    estimators = [
        ("xgb", Pipeline([("prep", preprocessor), ("model", XGBClassifier(**xgb_params))])),
        ("lgbm", Pipeline([("prep", preprocessor), ("model", LGBMClassifier(**lgbm_params))])),
    ]

    stacking = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=2000),
        stack_method="predict_proba",
        cv=5,
        n_jobs=-1,
    )

    return stacking


def run_oof_cv(model_builder, X, y, X_test, seed, model_tag):
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = model_builder(seed)
        model.fit(X_tr, y_tr)

        va_pred = model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = va_pred

        fold_auc = roc_auc_score(y_va, va_pred)
        fold_aucs.append(fold_auc)

        test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS

        print(f"[{model_tag}] seed={seed} fold={fold} AUC={fold_auc:.6f}")

    mean_auc = float(np.mean(fold_aucs))
    std_auc = float(np.std(fold_aucs))

    return oof_pred, test_pred, mean_auc, std_auc


In [ ]:
# === Prepare Data ===
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_test = test.copy()

# categorical columns (from original notebooks)
cat_cols = [
    "Chest pain type",
    "EKG results",
    "Slope of ST",
    "Thallium",
]

num_cols = [c for c in X.columns if c not in cat_cols]

print("Cat cols:", cat_cols)


In [ ]:
# === OOF: Stacking (non-leak) + Seed Ensemble ===

def stacking_builder(seed):
    return build_stacking_model(seed, ENCODER_MODE, cat_cols, num_cols)

stack_oof_all = []
stack_test_all = []
stack_seed_aucs = []

for seed in SEEDS:
    oof_pred, test_pred, mean_auc, std_auc = run_oof_cv(
        stacking_builder, X, y, X_test, seed, model_tag=f"stacking_{ENCODER_MODE}"
    )
    stack_oof_all.append(oof_pred)
    stack_test_all.append(test_pred)
    stack_seed_aucs.append(mean_auc)
    print(f"[stacking_{ENCODER_MODE}] seed={seed} OOF mean±std = {mean_auc:.6f} ± {std_auc:.6f}")

stack_oof = np.mean(stack_oof_all, axis=0)
stack_test = np.mean(stack_test_all, axis=0)

stack_oof_auc = roc_auc_score(y, stack_oof)
print(f"[stacking_{ENCODER_MODE}] FINAL OOF AUC (seed-avg): {stack_oof_auc:.6f}")

np.save(OUT_DIR / f"oof_pred_stacking_{ENCODER_MODE}.npy", stack_oof)
np.save(OUT_DIR / f"test_pred_stacking_{ENCODER_MODE}.npy", stack_test)


In [ ]:
# === OOF: CatBoost + Seed Ensemble ===
cat_cols_model = [
    "Sex",
    "FBS over 120",
    "Exercise angina",
    "EKG results",
    "Slope of ST",
    "Thallium",
    "Number of vessels fluro",
    "Chest pain type",
]

X_train_full = X.copy()
X_test_cb = X_test.copy()

# cast categorical to string
for c in cat_cols_model:
    if c in X_train_full.columns:
        X_train_full[c] = X_train_full[c].astype(str)
        X_test_cb[c] = X_test_cb[c].astype(str)

cat_idx = [X_train_full.columns.get_loc(c) for c in cat_cols_model if c in X_train_full.columns]

cat_oof_all = []
cat_test_all = []

for seed in SEEDS:
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(X_train_full))
    test_pred = np.zeros(len(X_test_cb))
    fold_aucs = []

    params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=1200,
        depth=5,
        learning_rate=0.08,
        random_seed=seed,
        verbose=False,
    )

    # try GPU, fallback CPU
    use_gpu = True
    if use_gpu:
        params.update(task_type="GPU", devices="0")

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_full, y), start=1):
        X_tr, X_va = X_train_full.iloc[tr_idx], X_train_full.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = CatBoostClassifier(**params)
        try:
            model.fit(
                X_tr,
                y_tr,
                cat_features=cat_idx,
                eval_set=(X_va, y_va),
                use_best_model=True,
                verbose=False,
            )
        except CatBoostError:
            cpu_params = {k: v for k, v in params.items() if k not in ["task_type", "devices"]}
            cpu_params["task_type"] = "CPU"
            model = CatBoostClassifier(**cpu_params)
            model.fit(
                X_tr,
                y_tr,
                cat_features=cat_idx,
                eval_set=(X_va, y_va),
                use_best_model=True,
                verbose=False,
            )

        va_pred = model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = va_pred

        fold_auc = roc_auc_score(y_va, va_pred)
        fold_aucs.append(fold_auc)
        print(f"[catboost] seed={seed} fold={fold} AUC={fold_auc:.6f}")

        test_pred += model.predict_proba(X_test_cb)[:, 1] / N_SPLITS

    mean_auc = float(np.mean(fold_aucs))
    std_auc = float(np.std(fold_aucs))
    print(f"[catboost] seed={seed} OOF mean±std = {mean_auc:.6f} ± {std_auc:.6f}")

    cat_oof_all.append(oof_pred)
    cat_test_all.append(test_pred)

cat_oof = np.mean(cat_oof_all, axis=0)
cat_test = np.mean(cat_test_all, axis=0)

cat_oof_auc = roc_auc_score(y, cat_oof)
print(f"[catboost] FINAL OOF AUC (seed-avg): {cat_oof_auc:.6f}")

np.save(OUT_DIR / "oof_pred_catboost.npy", cat_oof)
np.save(OUT_DIR / "test_pred_catboost.npy", cat_test)


In [ ]:
# === Blend: mean / rank mean / weighted grid ===

def rank_avg(a, b):
    ra = rankdata(a) / len(a)
    rb = rankdata(b) / len(b)
    return (ra + rb) / 2

mean_blend = (stack_oof + cat_oof) / 2
mean_auc = roc_auc_score(y, mean_blend)
print(f"[blend] mean OOF AUC: {mean_auc:.6f}")

rank_blend = rank_avg(stack_oof, cat_oof)
rank_auc = roc_auc_score(y, rank_blend)
print(f"[blend] rank-mean OOF AUC: {rank_auc:.6f}")

best_w = None
best_auc = -1
for w in np.arange(0.0, 1.0 + 1e-9, BLEND_GRID_STEP):
    blend = w * stack_oof + (1.0 - w) * cat_oof
    auc = roc_auc_score(y, blend)
    if auc > best_auc:
        best_auc = auc
        best_w = w

print(f"[blend] best weight: w={best_w:.2f}  OOF AUC={best_auc:.6f}")

# apply to test
final_test_blend = best_w * stack_test + (1.0 - best_w) * cat_test

submission = pd.DataFrame({
    "id": test.index,
    "Heart Disease": final_test_blend,
})
submission.to_csv("submission_blend.csv", index=False)
submission.head()


## Change Memo
- TargetEncoder moved inside each base model pipeline → no leakage in Stacking CV.
- Unified OOF CV with StratifiedKFold and seed averaging.
- CatBoost OOF aligned with same CV scheme + GPU fallback.
- Blending done on OOF (mean, rank-mean, weighted grid) and test submission generated.
